In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
import warnings
import sys
sys.path.append('..')
from config import *

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded!")

Libraries loaded!


In [2]:
df = pd.read_csv(RAW_DATA_PATH, na_values='?')
print(f"Raw data shape: {df.shape}")
print(f"Raw data loaded: {len(df):,} records")

Raw data shape: (101766, 50)
Raw data loaded: 101,766 records


In [3]:
# Columns to drop — reasons documented
DROP_COLS = [
    'encounter_id',      # ID — not a feature
    'patient_nbr',       # ID — not a feature
    'weight',            # 96.9% missing
    'max_glu_serum',     # 94.8% missing
    'A1Cresult',         # 83.3% missing
    'payer_code',        # 39.6% missing + not clinical
    'medical_specialty', # 49.1% missing
    # Drugs with near-zero variance
    'examide',
    'citoglipton',
    'glimepiride-pioglitazone',
    'metformin-rosiglitazone',
    'metformin-pioglitazone',
    'acetohexamide',
    'troglitazone',
    'tolazamide',
    'tolbutamide',
]

df.drop(columns=DROP_COLS, inplace=True)

print(f"After dropping columns: {df.shape}")
print(f"Columns dropped       : {len(DROP_COLS)}")
print(f"Remaining columns     : {df.shape[1]}")

After dropping columns: (101766, 34)
Columns dropped       : 16
Remaining columns     : 34


In [4]:
# Binary classification: early readmission (<30 days) = 1
# Clinical rationale: hospitals penalized for <30 day readmissions
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)
df.drop(columns=['readmitted'], inplace=True)

pos = df['readmitted_binary'].sum()
neg = len(df) - pos
print("TARGET: Early Readmission (<30 days)")
print("=" * 40)
print(f"Positive (readmitted <30) : {pos:,} ({pos/len(df)*100:.1f}%)")
print(f"Negative (not early)      : {neg:,} ({neg/len(df)*100:.1f}%)")
print(f"Imbalance ratio           : 1:{neg//pos:.0f}")
print("\nNote: SMOTE will be applied during modeling")

TARGET: Early Readmission (<30 days)
Positive (readmitted <30) : 11,357 (11.2%)
Negative (not early)      : 90,409 (88.8%)
Imbalance ratio           : 1:7

Note: SMOTE will be applied during modeling


In [5]:
print("BEFORE cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# race — fill with mode
df['race'].fillna(df['race'].mode()[0], inplace=True)

# diag_1, diag_2, diag_3 — fill with 'Unknown'
df['diag_1'].fillna('Unknown', inplace=True)
df['diag_2'].fillna('Unknown', inplace=True)
df['diag_3'].fillna('Unknown', inplace=True)

print("\nAFTER cleaning:")
remaining = df.isnull().sum()[df.isnull().sum() > 0]
if len(remaining) == 0:
    print("No missing values remaining!")
else:
    print(remaining)

BEFORE cleaning:
race      2273
diag_1      21
diag_2     358
diag_3    1423
dtype: int64

AFTER cleaning:
No missing values remaining!


In [6]:
# Remove duplicate patients (keep first encounter only)
before = len(df)
df.drop_duplicates(subset=['patient_nbr'] 
                   if 'patient_nbr' in df.columns 
                   else None, 
                   keep='first', inplace=True)
after = len(df)

# Remove invalid gender
df = df[df['gender'] != 'Unknown/Invalid']

print(f"Records before dedup : {before:,}")
print(f"Records after dedup  : {after:,}")
print(f"Records after gender : {len(df):,}")
print(f"Total removed        : {before - len(df):,}")

Records before dedup : 101,766
Records after dedup  : 101,766
Records after gender : 101,763
Total removed        : 3


In [7]:
# Age — convert to ordinal number
age_map = {
    '[0-10)'  : 0, '[10-20)' : 1, '[20-30)' : 2,
    '[30-40)' : 3, '[40-50)' : 4, '[50-60)' : 5,
    '[60-70)' : 6, '[70-80)' : 7, '[80-90)' : 8,
    '[90-100)': 9
}
df['age_numeric'] = df['age'].map(age_map)
df.drop(columns=['age'], inplace=True)

# Diagnosis — ICD-9 code grouping into disease categories
def categorize_diag(code):
    if pd.isna(code) or code == 'Unknown':
        return 'Unknown'
    code = str(code)
    if code.startswith('V') or code.startswith('E'):
        return 'External'
    try:
        code_num = float(code)
        if 390 <= code_num <= 459 or code_num == 785:
            return 'Circulatory'
        elif 460 <= code_num <= 519 or code_num == 786:
            return 'Respiratory'
        elif 520 <= code_num <= 579 or code_num == 787:
            return 'Digestive'
        elif 250 <= code_num <= 250.99:
            return 'Diabetes'
        elif 800 <= code_num <= 999:
            return 'Injury'
        elif 710 <= code_num <= 739:
            return 'Musculoskeletal'
        elif 580 <= code_num <= 629 or code_num == 788:
            return 'Genitourinary'
        elif 140 <= code_num <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except:
        return 'Other'

df['diag_1_cat'] = df['diag_1'].apply(categorize_diag)
df['diag_2_cat'] = df['diag_2'].apply(categorize_diag)
df['diag_3_cat'] = df['diag_3'].apply(categorize_diag)
df.drop(columns=['diag_1', 'diag_2', 'diag_3'], inplace=True)

# Drug change features
drug_cols = ['metformin', 'repaglinide', 'nateglinide',
             'chlorpropamide', 'glimepiride', 'glipizide',
             'glyburide', 'pioglitazone', 'rosiglitazone',
             'acarbose', 'miglitol', 'insulin',
             'glyburide-metformin', 'glipizide-metformin']

# Count drugs with dosage change
def drug_changed(val):
    return 1 if val in ['Up', 'Down'] else 0

def drug_on(val):
    return 0 if val == 'No' else 1

df['num_drugs_changed'] = df[drug_cols].apply(
    lambda row: sum(drug_changed(v) for v in row), axis=1)

df['num_drugs_active'] = df[drug_cols].apply(
    lambda row: sum(drug_on(v) for v in row), axis=1)

print("New features created:")
print(f"  age_numeric       : ordinal age (0-9)")
print(f"  diag_1/2/3_cat    : ICD-9 disease category")
print(f"  num_drugs_changed : drugs with dose change")
print(f"  num_drugs_active  : total active drugs")
print(f"\nCurrent shape: {df.shape}")

New features created:
  age_numeric       : ordinal age (0-9)
  diag_1/2/3_cat    : ICD-9 disease category
  num_drugs_changed : drugs with dose change
  num_drugs_active  : total active drugs

Current shape: (101763, 36)


In [8]:
# Binary columns — Yes/No → 1/0
binary_cols = ['change', 'diabetesMed', 'gender']
binary_map  = {'Yes': 1, 'No': 0, 'Ch': 1,
               'Female': 0, 'Male': 1}
for col in binary_cols:
    df[col] = df[col].map(binary_map)

# Drug columns — No/Steady/Up/Down → ordinal
drug_order  = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 2}
for col in drug_cols:
    if col in df.columns:
        df[col] = df[col].map(drug_order).fillna(0).astype(int)

# Race
le_race = LabelEncoder()
df['race'] = le_race.fit_transform(df['race'])

# Diagnosis categories
le_diag = LabelEncoder()
for col in ['diag_1_cat', 'diag_2_cat', 'diag_3_cat']:
    df[col] = le_diag.fit_transform(df[col])

# Remaining object columns
for col in df.select_dtypes('object').columns:
    if col != 'readmitted_binary':
        df[col] = LabelEncoder().fit_transform(
            df[col].astype(str))

print(f"After encoding: {df.shape}")
print(f"Remaining object cols: "
      f"{df.select_dtypes('object').columns.tolist()}")
print(f"\nAll dtypes:")
print(df.dtypes.value_counts())

After encoding: (101763, 36)
Remaining object cols: []

All dtypes:
int64    36
Name: count, dtype: int64


In [9]:
X = df.drop(columns=['readmitted_binary'])
y = df['readmitted_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# Save processed data
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv( PROCESSED_DIR / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv( PROCESSED_DIR / 'y_test.csv',  index=False)

print("PREPROCESSING COMPLETE")
print("=" * 45)
print(f"Final features    : {X.shape[1]}")
print(f"Training samples  : {len(X_train):,}")
print(f"Test samples      : {len(X_test):,}")
print()
print(f"Class distribution (train):")
vc = y_train.value_counts()
print(f"  Not early readmitted : {vc[0]:,} ({vc[0]/len(y_train)*100:.1f}%)")
print(f"  Early readmitted     : {vc[1]:,} ({vc[1]/len(y_train)*100:.1f}%)")
print()
print(f"Saved to: {PROCESSED_DIR}")
print()
print("Next Step: 03_modeling.ipynb")

PREPROCESSING COMPLETE
Final features    : 35
Training samples  : 81,410
Test samples      : 20,353

Class distribution (train):
  Not early readmitted : 72,324 (88.8%)
  Early readmitted     : 9,086 (11.2%)

Saved to: d:\Projects\healthcare-readmission-predictor\notebooks\..\data\processed

Next Step: 03_modeling.ipynb
